In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Tentukan lokasi folder dataset di laptopmu
BASE_DIR = 'rock-paper-scissors' # Sesuaikan jika nama foldermu berbeda
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'validation')
TEST_DIR = os.path.join(BASE_DIR, 'test')

print("TensorFlow Berhasil Dimuat, Versi:", tf.__version__)

In [ ]:
# Urutan alfabet folder: paper, rock, scissors
classes = ['paper', 'rock', 'scissors']

print("--- Pengecekan Jumlah Data Lokal ---")
for c in classes:
    train_count = len(os.listdir(os.path.join(TRAIN_DIR, c)))
    val_count = len(os.listdir(os.path.join(VAL_DIR, c)))
    test_count = len(os.listdir(os.path.join(TEST_DIR, c)))
    print(f"Kelas {c.upper()} -> Train: {train_count} | Val: {val_count} | Test: {test_count}")

# Menampilkan 3 sampel acak gambar dari laptop
plt.figure(figsize=(10, 4))
for i, c in enumerate(classes):
    class_path = os.path.join(TRAIN_DIR, c)
    random_img = random.choice(os.listdir(class_path))
    img = Image.open(os.path.join(class_path, random_img))
    
    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.title(f"Sampel: {c.upper()}")
    plt.axis('off')
plt.suptitle("EDA: Visualisasi Sampel Dataset Lokal")
plt.show()

In [ ]:
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32

# 1. Konfigurasi Augmentasi untuk Data Training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 2. Untuk Validation & Test CUKUP DI-RESCALE (Tidak boleh di-augmentasi)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# 3. Alirkan data dari folder laptop ke dalam program
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Wajib False untuk kebutuhan Evaluation/Confusion Matrix nanti
)

In [ ]:
# 1. Mengambil otak pre-trained model MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
base_model.trainable = False # Kunci bobot dasar agar tidak berubah

# 2. Menyusun layer classifier baru di atasnya
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),                 # Mencegah overfitting di laptop
    Dense(3, activation='softmax') # Output 3 kelas
])

# 3. Compile Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
EPOCHS = 30 # Naikkan ke 5 jika laptopmu memiliki GPU/VGA terdedikasi

print("--- ⏳ Memulai Proses Training di Laptop ---")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    verbose=1
)
print("--- Training Selesai! ---")

In [ ]:
# Evaluasi 1: Menampilkan Grafik Akurasi saat Training
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', marker='o')
plt.title('Grafik Akurasi Model')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()

# Evaluasi 2: Pengujian menggunakan Data TEST & Membuat Confusion Matrix
print("\n--- 🧠 Mengevaluasi Model dengan Data Test Baru ---")
Y_pred = model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

# Membuat plot Confusion Matrix hangat (Heatmap)
cm = confusion_matrix(y_true, y_pred)
plt.subplot(1, 2, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix (Data Test)')
plt.ylabel('Label Sebenarnya')
plt.xlabel('Hasil Prediksi Model')
plt.tight_layout()
plt.show()

# Menampilkan Classification Report angka presisi
print("\n--- 📈 Laporan Klasifikasi Detail ---")
print(classification_report(y_true, y_pred, target_names=classes))